# <center>2026 VIPER PTA Summer School<br><br>Author: Nima Laal

# Loading Packages

In [ ]:
import numpy as np

from NimaDemoUtils import *
import corner, warnings, pickle, copy

import jax.numpy as jnp
import jax.random as jrandom
import jax
import numpyro
import numpyro.distributions as dist
jax.config.update("jax_enable_x64", True)
from enterprise.signals import signal_base, gp_signals
from enterprise_extensions import blocks

from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc

from matplotlib.lines import Line2D
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format='retina'

hist_settings = dict(
    bins = 'auto',
    histtype = 'step', 
    lw = 3,
    density = True
)

plt.style.use('default')
def figsize(scale, wc = 1, hc = 1):
    fig_width_pt = 513.17 #469.755                  # Get this from LaTeX using \the\textwidth
    inches_per_pt = 1.0/72.27                       # Convert pt to inch
    golden_mean = (np.sqrt(5.0)-1.0)/2.0            # Aesthetic ratio (you could change this)
    fig_width = fig_width_pt*inches_per_pt*scale    # width in inches
    fig_height = fig_width*golden_mean              # height in inches
    fig_size = [wc * fig_width,hc * fig_height]
    return fig_size
plt.rcParams.update(plt.rcParamsDefault)

params = {#'backend': 'pdf',
        'axes.labelsize': 12,
        'lines.markersize': 4,
        'font.size': 10,
        'xtick.major.size':6,
        "xtick.top": True,
        "ytick.right": True,
        "xtick.minor.visible": True,
        "xtick.major.top": True, 
        "xtick.minor.top": True,
        "ytick.minor.visible": True, 
        "ytick.major.right": True, 
        "ytick.minor.right": True,
        "ytick.direction": "in",
        "xtick.direction": "in",
        'xtick.minor.size':3,  
        'ytick.major.size':6,
        'ytick.minor.size':3, 
        'xtick.major.width':0.5,
        'ytick.major.width':0.5,
        'xtick.minor.width':0.5,
        'ytick.minor.width':0.5,
        'lines.markeredgewidth':1,
        'axes.linewidth':1.2,
        'legend.fontsize': 7,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'savefig.dpi':200,
        'path.simplify':True,
        'font.family': 'serif',
        'font.serif':'Times',
        # "text.usetex": True,
        #'text.latex.preamble': [r'\usepackage{amsmath}'],
        'text.usetex':False,
        'figure.figsize': figsize(0.5, 1, 1)}
plt.rcParams.update(params)
plt.rcParams['font.family'] = 'STIXGeneral'  # Closely matches Computer Modern
plt.rcParams['mathtext.fontset'] = 'stix'    # Use STIX for math
plt.style.use('dark_background')

In [ ]:
def make_corner(chains, 
                colors, 
                labels = None, 
                ranges = None,
                truths = None,
                save_path = None):
    
    levels = [1 - np.exp(-0.5), 1 - np.exp(-1.0), 1 - np.exp(-1.5)]

    s = int((chains[0].shape[-1])/2)
    plt.close()
    fig = plt.figure(figsize = figsize(0.5, s, s))
    for chain, color in zip(chains, colors):
        fig = corner.corner(
        chain,
        fig=fig,
        range = ranges,
        labels=labels,
        color=color,
        bins=20,
        hist_bin_factor=2,
        truths = truths,
        truth_color = 'red',
        data_kwargs={'ms': 3},
        hist_kwargs={'density': True, 'lw': 2},
        contour_kwargs={'linewidths': 2},
        density=True,
        plot_datapoints=False,
        show_titles = True,
        levels = levels
    )


    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    plt.show()

# <center> The Simplest Case of a PTA!

## Making a PTA

In [ ]:
Npulsars = 4
n_realization = 1
seeds = np.linspace(10000, 10062300, n_realization, dtype = int)
disttype = 'uniform'
start_obs = 53000 #mjd units
dur_in_year = 13
end_obs = start_obs + dur_in_year * 365.25
toas = np.array([np.arange(start_obs,end_obs, 30)] * Npulsars)
toas

In [ ]:
lam, beta, pname, xiMat = PulsarDistMaker(Npulsars = Npulsars, 
                                          seed = seeds[0], 
                                          skyplot = True, 
                                          disttype = disttype)

## Injecting a GWB

In [ ]:
res_gw = np.zeros((n_realization, Npulsars, len(toas[0])))
for ii, seed in enumerate(seeds):
    print('Making Realization {0}/{1}...'.format(ii + 1, n_realization), end = '\r')
    res_gw[ii,:,:] = GWBInj(Amp = 1e-14,
           start_obs = start_obs, 
           end_obs = end_obs, 
           Npulsars = Npulsars, 
           ang = xiMat.flatten(), 
           seed = seed, 
           toas = toas)

In [ ]:
plt.figure(figsize=figsize(0.5, 2, 2))
pulsar_index = 0
realz_index = 0
plt.plot(toas[0], res_gw[realz_index, pulsar_index, :])
plt.xlabel('TOAS (MJD)')
plt.ylabel('Red Noise Residuals')
plt.show()

## Add White Noise to Simulate Pulsar TOA Error

In [ ]:
wn_level_in_second = 1e-7
white_noise = np.random.normal(loc = 0, scale = wn_level_in_second, size = res_gw.shape)

In [ ]:
plt.figure(figsize=figsize(0.5, 2, 2))
plt.plot(toas[0], white_noise[realz_index, pulsar_index, :])
plt.xlabel('TOAS (MJD)')
plt.ylabel('White Residuals')
plt.show()

## Combine All

In [ ]:
total_res = res_gw + white_noise

In [ ]:
plt.figure(figsize=figsize(0.5, 2, 2))
plt.plot(toas[0], white_noise[realz_index, pulsar_index, :], color = 'white', label = 'white noise')
plt.plot(toas[0], res_gw[realz_index, pulsar_index, :], color = 'red', label = 'red noise')
plt.plot(toas[0], total_res[realz_index, pulsar_index, :], color = 'gold', label = 'total noise (observed TOAs - true timing model)')
plt.legend(fontsize = 14)
plt.xlabel('TOAS (MJD)')
plt.ylabel('Residuals')
plt.show()

# Constructing the Model

## The Basis Matrix

In [ ]:
def Fmat(freqs, toas):
    '''
    The 'F-matrix' used to do a discrete Fourier transform
    '''
    nmodes = len(freqs)
    N = len(toas)
    F = np.zeros((N, 2 * nmodes))
    F[:, 0::2] = np.sin(2 * np.pi * toas[:, None] * freqs[None, :])
    F[:, 1::2] = np.cos(2 * np.pi * toas[:, None] * freqs[None, :])
    return F

In [ ]:
Tspan = (toas.max() - toas.min()) * (60 * 60 * 24)
n_bins = 10
freqs = np.arange(1/Tspan, (n_bins+.01)/Tspan, 1/Tspan)
freqs 

In [ ]:
Fmatrix_single_pulsar = Fmat(freqs, toas[0] * (60 * 60 * 24))
Fmatrix_single_pulsar.shape

In [ ]:
Fmatrix_multi_pulsar = np.broadcast_to(Fmatrix_single_pulsar, (Npulsars, *Fmatrix_single_pulsar.shape))
Fmatrix_multi_pulsar.shape

In [ ]:
Fmatrix_multi_pulsar = jnp.concat(Fmatrix_multi_pulsar)
Fmatrix_multi_pulsar.shape

## Putting the Model Together

In [ ]:
total_res = jnp.concat(total_res[realz_index])
n_toas = total_res.shape[0]

In [ ]:
def model():

    ######################################## Prior ########################################
    half_log10_rho = numpyro.sample('related_to_char_strain', dist.Uniform(-9., -1.).expand([n_bins]))
    rho = jnp.repeat(10**(half_log10_rho), repeats=2)

    a_GWB = numpyro.sample('a_GWB', dist.Normal(0, rho))

    ######################################## PTA likelihood ########################################
    mean_model = Fmatrix_multi_pulsar @ a_GWB
    numpyro.sample(
        'lnlikelihood',
        dist.Normal(mean_model, wn_level_in_second),
        obs=total_res,
    )

## MCMC

In [ ]:
# nuts_kernel = numpyro.infer.NUTS(
#     model=model,
# )

# mcmc = numpyro.infer.MCMC(
#     sampler=nuts_kernel,
#     num_warmup=500,
#     num_samples=1000,
#     num_chains=1,
# )

# mcmc.run(jrandom.key(170817))
# samples = mcmc.get_samples()

![alt text](meme.jpg "PTMCMC")

## Anything wrong? Too slow?

# The Cure!

In [ ]:
def model():

    ######################################## Prior ########################################

    half_log10_rho = numpyro.sample('related_to_char_strain', dist.Uniform(-9., -1.).expand([n_bins]))
    rho = jnp.repeat(10**(half_log10_rho), repeats=2)
    phi = rho**2  # prior variance per Fourier coefficient, shape (2*n_bins,)

    ######################################## Marginal likelihood ########################################

    N = jnp.diag(jnp.ones(n_toas) * wn_level_in_second**2)                       # (n_toa, n_toa) -- or wn_level_in_second * jnp.eye(n_toa)

    cov = N + Fmatrix_multi_pulsar @ (phi[:, None] * Fmatrix_multi_pulsar.T)          # N + F @ diag(phi) @ F.T

    numpyro.sample(
        'lnlikelihood',
        dist.MultivariateNormal(loc=0, covariance_matrix=cov),
        obs=total_res,
    )

In [ ]:
nuts_kernel = numpyro.infer.NUTS(
    model=model,
)

mcmc = numpyro.infer.MCMC(
    sampler=nuts_kernel,
    num_warmup=500,
    num_samples=1000,
    num_chains=1,
)

mcmc.run(jrandom.key(170817))
samples = mcmc.get_samples()

In [ ]:
chain = mcmc.get_samples()['related_to_char_strain']
chain.shape

In [ ]:
YEAR = 365.25 * 24 * 3600.0

def powerlaw_to_free_spectrum(log10_A, gamma, freqs):
    """
    Convert a PTA power-law GWB into free-spectrum PSD values.

    Parameters
    ----------
    log10_A : float
        log10 strain amplitude at 1/year.
    gamma : float
        Spectral index.
    freqs : ndarray
        Fourier frequencies in Hz.

    Returns
    -------
    rho : ndarray
        PSD values at each frequency.
    """

    A = 10.0**log10_A
    fyr = 1.0 / YEAR

    rho = (
        A**2
        / (12.0 * np.pi**2 * Tspan *freqs**3)
        * (freqs / fyr) ** (3-gamma)
    )

    return rho

truths = 0.5 * np.log10(powerlaw_to_free_spectrum(
    log10_A=np.log10(1e-14),
    gamma=13/3,
    freqs=freqs,
))
truths

In [ ]:
make_corner(chains = [np.array(chain)], 
                colors = ['gold'], 
                labels = None, 
                ranges = None,
                truths = truths,
                save_path = None)

# Let's Use `enterprise`

## First let's turn our residual ist into somthing `enterprise` can understand!

In [ ]:
with open('../Data/donor_pickle.pkl', 'rb') as fin:
    psrs_donor = pickle.load(fin)[:Npulsars]
len(psrs_donor)

In [ ]:
def write_to_psrs(residual_list):
    ans = []
    psrs_copy = copy.deepcopy(psrs_donor)
    pulsar_positions = lam_beta_to_xyz(lam, beta)
    for pidx, psr in enumerate(psrs_copy):

        psr._toas = np.array(toas[pidx] * (60 * 60 * 24))
        psr._toaerrs = np.array(wn_level_in_second * np.ones_like(toas[pidx]))
        psr._residuals = np.array(residual_list[pidx])
        psr._pos = pulsar_positions[pidx]
        psr._designmatrix = np.array(Mmat(toas[pidx]))
        psr.sort_data()
    ans.append(psrs_copy)
    return ans[0]

In [ ]:
psrs = write_to_psrs(res_gw[0])[0:1]
len(psrs)

In [ ]:
noise_dict = {}
for pname in [psr.name for psr in psrs]:
    noise_dict.update({pname + '_efac': 1.0})
    noise_dict.update({pname + '_log10_t2equad': -np.inf})

In [ ]:
tm = gp_signals.MarginalizingTimingModel(use_svd=True)

wn = blocks.white_noise_block(
    vary=False,
    inc_ecorr=False,
    gp_ecorr=False,
    select='none',
    tnequad=False,
)

gwb = blocks.common_red_noise_block(
    psd="powerlaw",
    prior="log-uniform",
    Tspan=Tspan,
    components=n_bins,
    gamma_val=None,
    name="gw",
    orf="crn",
)

s = wn + gwb + tm

pta = signal_base.PTA(
    [s(p) for p in psrs], signal_base.LogLikelihoodDenseCholesky
)
pta.set_default_params(noise_dict)

In [ ]:
# set initial parameters drawn from prior
x0 = np.hstack([p.sample() for p in pta.params])
ndim = len(x0)

In [ ]:
cov = np.diag(np.ones(ndim) * 0.01**2)
# set the location to save the output chains
outDir = '../Chain/'
sampler = ptmcmc(ndim, pta.get_lnlikelihood, pta.get_lnprior, cov, 
                 outDir=outDir, resume=False)

In [ ]:
N = int(1e5)
x0 = np.hstack([p.sample() for p in pta.params])
sampler.sample(x0, N, SCAMweight=30, AMweight=15, DEweight=50, )

In [ ]:
chain = np.loadtxt(outDir + '/chain_1.txt')[:, :-4]
burn = int(0.25 * chain.shape[0])
chain = chain[burn:, :]
chain.shape

In [ ]:
make_corner(chains = [np.array(chain)], 
                colors = ['gold'], 
                labels = None, 
                ranges = None,
                truths = [13/3, np.log10(1e-14)],
                save_path = None)